# Variables climáticas por fase fenológica

Objetivo:
Construir una metodología reproducible para vincular los píxeles agrícolas utilizados en los análisis SST-rendimiento con variables climáticas locales derivadas de Earth Engine.

El flujo consiste en:

1. Identificar los píxeles válidos utilizados en las correlaciones SST-rendimiento.
2. Convertir los índices espaciales del H5 a coordenadas geográficas.
3. Reconstruir calendarios fenológicos típicos por cultivo y franja latitudinal.
4. Asociar cada píxel a una ventana fenológica F1-F3.
5. Preparar un plan de extracción climática para Earth Engine.

## Inicialización de Google Earth Engine

Se verifica la conexión a Earth Engine utilizando el proyecto asociado a la tesis.

In [3]:
import ee

ee.Authenticate(auth_mode="localhost")

ee.Initialize()

print(ee.String("Earth Engine OK").getInfo())


Successfully saved authorization token.


EEException: ee.Initialize: no project found. Call with project= or see http://goo.gle/ee-auth.

In [4]:
import ee

ee.Initialize(project="jovial-opus-394022")

print("Earth Engine OK")

Earth Engine OK


## Verificación de acceso a ERA5-Land

Se utiliza un píxel agrícola de prueba para verificar el acceso a las variables climáticas diarias.

In [6]:
import ee
import pandas as pd

ee.Initialize(project="jovial-opus-394022")  # o el ID que te funcionó

LAT = -40.25
LON = -72.75

pixel = ee.Geometry.Rectangle([
    LON - 0.25,
    LAT - 0.25,
    LON + 0.25,
    LAT + 0.25
])

START = "1983-01-01"
END = "1983-01-08"

era5 = (
    ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
    .filterDate(START, END)
)

print("Número de días:", era5.size().getInfo())

def extract_day(img):
    date = ee.Date(img.get("system:time_start"))

    stats = img.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=pixel,
        scale=10000,
        maxPixels=1e9
    )

    return ee.Feature(None, {
        "date": date.format("YYYY-MM-dd"),
        "tmean_c": ee.Number(stats.get("temperature_2m")).subtract(273.15),
        "tmax_c": ee.Number(stats.get("temperature_2m_max")).subtract(273.15),
        "tmin_c": ee.Number(stats.get("temperature_2m_min")).subtract(273.15),
        "precip_mm": ee.Number(stats.get("total_precipitation_sum")).multiply(1000),
        "swvl1": stats.get("volumetric_soil_water_layer_1"),
        "swvl2": stats.get("volumetric_soil_water_layer_2"),
        "swvl3": stats.get("volumetric_soil_water_layer_3")
    })

table = era5.map(extract_day)

features = table.getInfo()["features"]
df = pd.DataFrame([f["properties"] for f in features]).sort_values("date")

display(df)

df.to_csv("pixel_chile_era5_test_7days.csv", index=False, encoding="utf-8-sig")
print("CSV guardado: pixel_chile_era5_test_7days.csv")

Número de días: 7


,date,precip_mm,swvl1,swvl2,swvl3,tmax_c,tmean_c,tmin_c
0,1983-01-01,0.002967,0.152656,0.194828,0.264849,22.855214,17.132219,11.838251
1,1983-01-02,0.130594,0.150221,0.191397,0.262014,25.499269,17.636853,11.760756
2,1983-01-03,0.373843,0.148797,0.188043,0.259068,30.387949,21.279722,13.895335
3,1983-01-04,0.009624,0.148417,0.184625,0.255894,28.517399,22.132835,14.326503
4,1983-01-05,1.224635,0.148712,0.181951,0.253200,23.283038,17.760317,13.948497
5,1983-01-06,0.000201,0.154287,0.181040,0.251929,23.496356,17.643067,11.149455
6,1983-01-07,0.520959,0.152800,0.178729,0.249330,20.628062,16.388037,12.813405


CSV guardado: pixel_chile_era5_test_7days.csv


## Diagnóstico de la tabla maestra

Se inspecciona la estructura fenológica utilizada posteriormente para construir calendarios típicos por cultivo y región.

In [8]:
import pandas as pd

archivo = r"C:\Users\paola\Tesis\03_Resultados\Fenologia\geoglam_cm4ew_tabla_maestra.xlsx"

xls = pd.ExcelFile(archivo)

print(xls.sheet_names)

for hoja in xls.sheet_names:
    df = pd.read_excel(archivo, sheet_name=hoja)

    print("\n")
    print("="*60)
    print(hoja)
    print(df.shape)
    print(df.columns.tolist())

['tabla_maestra', 'Hoja1', 'Hoja3']


tabla_maestra
(4415, 44)
['record_id', 'cultivo', 'fase_original', 'fase_estandar', 'pais', 'region', 'temporada', 'lat', 'lon', 'phase_id', 'phase_order', 'phase_code', 'phase_name', 'phase_start_label', 'phase_end_label', 'phase_start_doy', 'phase_end_doy', 'phase_duration', 'wraps_year', 'culti', 'country', 'crop_original', 'planting', 'vegetative', 'harvest', 'endofseason', 'duration_vegetative', 'duration_hasta_cosecha', 'duration_total', 'outofseason', 'minimalpro', 'variable_critica', 'umbral_optimo', 'umbral', 'tipo_estres', 'impacto_cultivo', 'nivel_evidencia', 'fuente', 'enlace', 'cultivo_normalizado', 'matched_sensib_clima', 'matched_fenologia', 'source_merge_status', 'data_completeness']


Hoja1
(19, 9)
['Cultivo', 'Fase fisiológica reportada', 'Variable de amenaza', 'Umbral reportado', 'Impacto reportado', 'Fragmento exacto (inglés) para rastreo', 'Autor principal', 'Año', 'Título del artículo']


Hoja3
(8, 4)
['Variable climática crít

# Diagnóstico base antes de extraer variables climáticas

Este bloque prepara la base espacial y fenológica antes de consultar Google Earth Engine.

Se realizan cuatro tareas:

1. Revisar la estructura de la tabla maestra fenológica.
2. Inspeccionar los archivos H5 de correlación SST-rendimiento.
3. Convertir `lat_idx` y `lon_idx` de los H5 a coordenadas reales usando la grilla SST 0.5°.
4. Guardar el inventario de píxeles válidos por cultivo en la carpeta de resultados.

Este paso no descarga clima desde Earth Engine. Solo define cuáles son los píxeles agrícolas válidos que posteriormente podrán cruzarse con variables climáticas por fase fenológica.

In [17]:
# ============================================================
# DIAGNÓSTICO BASE: TABLA MAESTRA + H5 SST-RENDIMIENTO
# Antes de extraer clima desde Google Earth Engine
# ============================================================

from pathlib import Path
import h5py
import xarray as xr
import pandas as pd
import numpy as np
import math

# ============================================================
# 1. RUTAS
# ============================================================

OUT_DIR = Path(r"C:\Users\paola\Tesis\03_Resultados\Clima_phase")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MASTER_XLSX = Path(
    r"C:\Users\paola\Tesis\03_Resultados\Fenologia\geoglam_cm4ew_tabla_maestra.xlsx"
)

SST_GRID = Path(r"D:\SST_monthly_detrended_1981_2025_0p5deg35.nc")

H5_FILES = {
    "wheat": Path(r"D:\wheat_correlacion_vectorizada.h5"),
    "maize": Path(r"D:\maize_correlacion_vectorizada.h5"),
    "rice": Path(r"D:\rice_correlacion_vectorizada.h5"),
    "soybean": Path(r"D:\soybean_correlacion_vectorizada.h5"),
}

# Salidas
OUT_MASTER_DIAG = OUT_DIR / "diagnostico_tabla_maestra.csv"
OUT_H5_META = OUT_DIR / "diagnostico_h5_metadata.csv"
OUT_PIXELS_ALL = OUT_DIR / "all_h5_pixels_latlon.csv"
OUT_PIXELS_BY_BAND = OUT_DIR / "resumen_pixeles_h5_por_cultivo_franja.csv"

# ============================================================
# 2. FUNCIONES AUXILIARES
# ============================================================

def fixed_band(lat: float) -> str:
    """
    Asigna una latitud a una franja de 10°.
    Ejemplos:
    52.25  -> 60_to_50
    34.75  -> 40_to_30
    5.25   -> 10_to_0
    -4.75  -> 0_to_-10
    -40.25 -> -40_to_-50
    """
    upper = math.ceil(float(lat) / 10) * 10

    if float(lat) == upper:
        upper += 10

    lower = upper - 10
    return f"{int(upper)}_to_{int(lower)}"


def inspect_h5(path: Path, crop: str) -> pd.DataFrame:
    """
    Inspecciona datasets internos de un H5.
    """
    rows = []

    with h5py.File(path, "r") as f:
        def visitor(name, obj):
            if isinstance(obj, h5py.Dataset):
                rows.append({
                    "crop": crop,
                    "file": str(path),
                    "dataset": name,
                    "shape": str(obj.shape),
                    "dtype": str(obj.dtype)
                })

        f.visititems(visitor)

    return pd.DataFrame(rows)


def load_h5_pixels(path: Path, crop: str, lats: np.ndarray, lons: np.ndarray) -> pd.DataFrame:
    """
    Convierte lat_idx y lon_idx del H5 a coordenadas reales usando
    la grilla SST 0.5°.
    """
    with h5py.File(path, "r") as f:
        lat_idx = f["lat_idx"][:]
        lon_idx = f["lon_idx"][:]

    df = pd.DataFrame({
        "crop": crop,
        "pixel_id_h5": np.arange(len(lat_idx)),
        "lat_idx": lat_idx,
        "lon_idx": lon_idx,
        "lat": lats[lat_idx],
        "lon": lons[lon_idx],
    })

    # Longitud compatible con Earth Engine
    df["lon_ee"] = np.where(df["lon"] > 180, df["lon"] - 360, df["lon"])

    # Franja latitudinal de 10°
    df["lat_band"] = df["lat"].apply(fixed_band)

    # Polígono GDHY 0.5° para extracción posterior en GEE
    df["pixel_lat_min"] = df["lat"] - 0.25
    df["pixel_lat_max"] = df["lat"] + 0.25
    df["pixel_lon_min_ee"] = df["lon_ee"] - 0.25
    df["pixel_lon_max_ee"] = df["lon_ee"] + 0.25

    return df


# ============================================================
# 3. DIAGNÓSTICO DE TABLA MAESTRA
# ============================================================

xls = pd.ExcelFile(MASTER_XLSX)

print("Hojas disponibles en tabla maestra:")
print(xls.sheet_names)

master = pd.read_excel(MASTER_XLSX, sheet_name="tabla_maestra")

print("\nDimensiones tabla_maestra:")
print(master.shape)

print("\nColumnas tabla_maestra:")
print(master.columns.tolist())

master_diag = pd.DataFrame({
    "columna": master.columns,
    "n_no_nulos": [master[c].notna().sum() for c in master.columns],
    "n_unicos": [master[c].nunique(dropna=True) for c in master.columns],
    "tipo": [str(master[c].dtype) for c in master.columns],
})

master_diag.to_csv(OUT_MASTER_DIAG, index=False, encoding="utf-8-sig")

print("\nCultivos en tabla maestra:")
print(master["cultivo"].value_counts(dropna=False))

if {"cultivo", "lat", "lon"}.issubset(master.columns):
    print("\nPíxeles únicos en tabla maestra por cultivo:")
    display(
        master[["cultivo", "lat", "lon"]]
        .drop_duplicates()
        .groupby("cultivo")
        .size()
        .reset_index(name="n_pixeles_tabla_maestra")
    )

if {"cultivo", "phase_duration"}.issubset(master.columns):
    print("\nDuración de fases en tabla maestra:")
    display(
        master.groupby("cultivo")["phase_duration"]
        .describe()
        .reset_index()
    )

print(f"\nDiagnóstico tabla maestra guardado en:\n{OUT_MASTER_DIAG}")

# ============================================================
# 4. LEER GRILLA SST 0.5°
# ============================================================

ds_sst = xr.open_dataset(SST_GRID)

lat_name = "lat" if "lat" in ds_sst.coords else "latitude"
lon_name = "lon" if "lon" in ds_sst.coords else "longitude"

lats = ds_sst[lat_name].values
lons = ds_sst[lon_name].values

print("\nGrilla SST usada para convertir índices H5:")
print(ds_sst)

print("\nLat shape:", lats.shape)
print("Lon shape:", lons.shape)
print("Lat ejemplo:", lats[:5], "...", lats[-5:])
print("Lon ejemplo:", lons[:5], "...", lons[-5:])

# ============================================================
# 5. INSPECCIÓN H5 Y CONVERSIÓN DE ÍNDICES A COORDENADAS
# ============================================================

all_meta = []
all_pixels = []

for crop, h5_path in H5_FILES.items():

    print("\n" + "=" * 80)
    print(f"Procesando H5: {crop}")
    print(h5_path)
    print("=" * 80)

    if not h5_path.exists():
        print(f"ERROR: no existe el archivo {h5_path}")
        continue

    meta = inspect_h5(h5_path, crop)
    all_meta.append(meta)

    display(meta)

    required = set(meta["dataset"].tolist())
    if not {"lat_idx", "lon_idx"}.issubset(required):
        print(f"ERROR: {crop} no contiene lat_idx/lon_idx.")
        continue

    pixels_crop = load_h5_pixels(h5_path, crop, lats, lons)

    out_crop = OUT_DIR / f"{crop}_h5_pixels_latlon.csv"
    pixels_crop.to_csv(out_crop, index=False, encoding="utf-8-sig")

    print(f"Píxeles reconstruidos para {crop}: {len(pixels_crop)}")
    print(f"Guardado: {out_crop}")

    all_pixels.append(pixels_crop)

# ============================================================
# 6. GUARDAR INVENTARIOS CONSOLIDADOS
# ============================================================

h5_meta = pd.concat(all_meta, ignore_index=True)
h5_meta.to_csv(OUT_H5_META, index=False, encoding="utf-8-sig")

pixels_all = pd.concat(all_pixels, ignore_index=True)
pixels_all.to_csv(OUT_PIXELS_ALL, index=False, encoding="utf-8-sig")

summary_band = (
    pixels_all
    .groupby(["crop", "lat_band"])
    .size()
    .reset_index(name="n_pixels")
    .sort_values(["crop", "lat_band"])
)

summary_band.to_csv(OUT_PIXELS_BY_BAND, index=False, encoding="utf-8-sig")

# ============================================================
# 7. RESÚMENES FINALES
# ============================================================

print("\n" + "=" * 80)
print("RESUMEN FINAL")
print("=" * 80)

print("\nPíxeles H5 válidos por cultivo:")
print(pixels_all.groupby("crop").size())

print("\nTotal píxeles H5 válidos:")
print(len(pixels_all))

print("\nPíxeles por cultivo y franja latitudinal:")
display(summary_band)

print("\nArchivos generados:")
print(OUT_MASTER_DIAG)
print(OUT_H5_META)
print(OUT_PIXELS_ALL)
print(OUT_PIXELS_BY_BAND)

display(pixels_all.head())

Hojas disponibles en tabla maestra:
['tabla_maestra', 'Hoja1', 'Hoja3']

Dimensiones tabla_maestra:
(4415, 44)

Columnas tabla_maestra:
['record_id', 'cultivo', 'fase_original', 'fase_estandar', 'pais', 'region', 'temporada', 'lat', 'lon', 'phase_id', 'phase_order', 'phase_code', 'phase_name', 'phase_start_label', 'phase_end_label', 'phase_start_doy', 'phase_end_doy', 'phase_duration', 'wraps_year', 'culti', 'country', 'crop_original', 'planting', 'vegetative', 'harvest', 'endofseason', 'duration_vegetative', 'duration_hasta_cosecha', 'duration_total', 'outofseason', 'minimalpro', 'variable_critica', 'umbral_optimo', 'umbral', 'tipo_estres', 'impacto_cultivo', 'nivel_evidencia', 'fuente', 'enlace', 'cultivo_normalizado', 'matched_sensib_clima', 'matched_fenologia', 'source_merge_status', 'data_completeness']

Cultivos en tabla maestra:
cultivo
maize      1490
rice       1060
wheat       780
sorghum     625
millet      250
beans       170
teff         40
Name: count, dtype: int64

Píxel

,cultivo,n_pixeles_tabla_maestra
0,beans,14
1,maize,224
2,millet,50
3,rice,139
4,sorghum,116
5,teff,8
6,wheat,143



Duración de fases en tabla maestra:


,cultivo,count,mean,std,min,25%,50%,75%,max
0,beans,170.0,73.000000,74.774770,15.0,30.0,45.0,61.0,244.0
1,maize,1490.0,73.000000,53.951910,14.0,31.0,60.0,92.0,242.0
2,millet,250.0,73.000000,49.193822,16.0,31.0,61.0,92.0,212.0
3,rice,1060.0,73.000000,47.858463,13.0,31.0,61.0,92.0,273.0
4,sorghum,625.0,73.000000,47.430516,15.0,31.0,61.0,92.0,227.0
5,teff,40.0,73.000000,29.943537,31.0,61.0,62.0,104.0,122.0
6,wheat,780.0,73.737179,47.024631,15.0,31.0,61.0,92.0,243.0



Diagnóstico tabla maestra guardado en:
C:\Users\paola\Tesis\03_Resultados\Clima_phase\diagnostico_tabla_maestra.csv

Grilla SST usada para convertir índices H5:
<xarray.Dataset> Size: 1GB
Dimensions:        (lat: 360, lon: 720, time: 529)
Coordinates:
  * lat            (lat) float32 1kB -89.75 -89.25 -88.75 ... 88.75 89.25 89.75
  * lon            (lon) float32 3kB 0.25 0.75 1.25 1.75 ... 358.8 359.2 359.8
  * time           (time) datetime64[ns] 4kB 1981-09-01 ... 2025-09-01
Data variables:
    sst_detrended  (lat, lon, time) float64 1GB ...

Lat shape: (360,)
Lon shape: (720,)
Lat ejemplo: [-89.75 -89.25 -88.75 -88.25 -87.75] ... [87.75 88.25 88.75 89.25 89.75]
Lon ejemplo: [0.25 0.75 1.25 1.75 2.25] ... [357.75 358.25 358.75 359.25 359.75]

Procesando H5: wheat
D:\wheat_correlacion_vectorizada.h5


,crop,file,dataset,shape,dtype
0,wheat,D:\wheat_correlacion_vectorizada.h5,correlacion,"(10505, 12, 259200)",float32
1,wheat,D:\wheat_correlacion_vectorizada.h5,lat_idx,"(10505,)",int64
2,wheat,D:\wheat_correlacion_vectorizada.h5,lon_idx,"(10505,)",int64


... (6 outputs left out)

Píxeles reconstruidos para soybean: 4955
Guardado: C:\Users\paola\Tesis\03_Resultados\Clima_phase\soybean_h5_pixels_latlon.csv

RESUMEN FINAL

Píxeles H5 válidos por cultivo:
crop
maize      10923
rice        3186
soybean     4955
wheat      10505
dtype: int64

Total píxeles H5 válidos:
29569

Píxeles por cultivo y franja latitudinal:


,crop,lat_band,n_pixels
0,maize,-10_to_-20,784
1,maize,-20_to_-30,698
2,maize,-30_to_-40,221
3,maize,-40_to_-50,1
4,maize,0_to_-10,938
5,maize,10_to_0,1344
6,maize,20_to_10,1255
7,maize,30_to_20,1017
8,maize,40_to_30,2256
9,maize,50_to_40,2379



Archivos generados:
C:\Users\paola\Tesis\03_Resultados\Clima_phase\diagnostico_tabla_maestra.csv
C:\Users\paola\Tesis\03_Resultados\Clima_phase\diagnostico_h5_metadata.csv
C:\Users\paola\Tesis\03_Resultados\Clima_phase\all_h5_pixels_latlon.csv
C:\Users\paola\Tesis\03_Resultados\Clima_phase\resumen_pixeles_h5_por_cultivo_franja.csv


,crop,pixel_id_h5,lat_idx,lon_idx,lat,lon,lon_ee,lat_band,pixel_lat_min,pixel_lat_max,pixel_lon_min_ee,pixel_lon_max_ee
0,wheat,0,94,575,-42.75,287.75,-72.25,-40_to_-50,-43.0,-42.5,-72.5,-72.0
1,wheat,1,97,573,-41.25,286.75,-73.25,-40_to_-50,-41.5,-41.0,-73.5,-73.0
2,wheat,2,97,575,-41.25,287.75,-72.25,-40_to_-50,-41.5,-41.0,-72.5,-72.0
3,wheat,3,98,573,-40.75,286.75,-73.25,-40_to_-50,-41.0,-40.5,-73.5,-73.0
4,wheat,4,98,574,-40.75,287.25,-72.75,-40_to_-50,-41.0,-40.5,-73.0,-72.5


# Construcción del plan de extracción climática por fase fenológica

El objetivo de esta etapa es transformar el inventario espacial de píxeles agrícolas en un conjunto de consultas climáticas reproducibles para Google Earth Engine.

Para ello se utilizarán los calendarios fenológicos típicos generados previamente a partir de la tabla maestra. Cada píxel será asociado a:

- Cultivo.
- Franja latitudinal de 10°.
- Temporada agrícola.
- Fase fenológica (F1, F2 y F3).
- Día de inicio de la fase.
- Día de finalización de la fase.
- Duración de la fase.

El resultado final será una tabla de planificación de extracción (`phase_extraction_plan.csv`) que permitirá consultar variables climáticas únicamente durante las ventanas fenológicas relevantes, evitando la descarga innecesaria de series diarias completas.

Esta estrategia reduce significativamente el volumen de datos a procesar y garantiza la consistencia espacial entre:

- Rendimiento agrícola (GDHY).
- Correlaciones SST-rendimiento.
- Calendarios fenológicos típicos.
- Variables climáticas extraídas desde Earth Engine.

A partir de este plan se construirán posteriormente los indicadores climáticos por fase, incluyendo temperatura, precipitación, humedad del suelo, evapotranspiración y métricas de estrés térmico e hídrico.

# Resultado de la preparación para la calculadora paramétrica

Las pruebas realizadas en esta etapa permitieron verificar la disponibilidad de variables climáticas diarias en Google Earth Engine y construir la infraestructura espacial necesaria para realizar consultas bajo demanda.

Se comprobó el acceso a ERA5-Land Daily Aggregated y la disponibilidad de variables relevantes para el análisis agroclimático, incluyendo temperatura máxima, media y mínima, precipitación y humedad de suelo.

Adicionalmente, se reconstruyó la ubicación geográfica de todos los píxeles agrícolas utilizados en los análisis SST-rendimiento mediante la conversión de los índices espaciales almacenados en los archivos H5 a coordenadas geográficas compatibles con la grilla SST de 0.5°.

Como resultado se generó un inventario consolidado de 29.569 píxeles agrícolas válidos distribuidos entre maíz, arroz, trigo y soya, incluyendo:

- Coordenadas geográficas.
- Franja latitudinal de 10°.
- Geometría GDHY equivalente (0.5° × 0.5°).
- Identificadores espaciales originales.

Esta información constituye la capa de enlace entre:

- Rendimiento agrícola (GDHY).
- Correlaciones SST-rendimiento.
- Calendarios fenológicos típicos.
- Variables climáticas consultadas desde Google Earth Engine.

La siguiente etapa del trabajo no consiste en descargar masivamente información climática, sino en utilizar este inventario espacial junto con los calendarios fenológicos típicos para construir una calculadora paramétrica capaz de consultar series diarias bajo demanda, identificar eventos críticos, calcular excedencias de umbrales climáticos y estimar probabilidades históricas de ocurrencia para aplicaciones orientadas al análisis de riesgo y al diseño de seguros paramétricos agrícolas.